# Generate OTTO submission.csv

Notebook này tải model SASRec đã huấn luyện, sinh dự đoán cho tập test OTTO và lưu ra `submission.csv`.

Bạn có thể chỉnh `checkpoint_path`, `test_jsonl_path` và `submission_path` cho phù hợp.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch

In [ ]:
# Paths - cập nhật nếu cần
checkpoint_path = Path('best_sasrec_model.ckpt')
candidate_paths = [
    Path('/kaggle/input/datasets/organizations/otto/recsys-dataset/otto-recsys-test.jsonl'),
    Path('datasets/otto/otto-recsys-test.jsonl'),
    Path('datasets/otto-recsys-test.jsonl'),
    Path('otto-recsys-test.jsonl')
]
test_jsonl_path = next((p for p in candidate_paths if p.exists()), None)
if test_jsonl_path is None:
    raise FileNotFoundError('Không tìm thấy file test JSONL. Hãy đặt đúng đường dẫn trong `candidate_paths`.')

submission_path = Path('submission.csv')
print(f'Using checkpoint: {checkpoint_path}')
print(f'Using test file: {test_jsonl_path}')
print(f'Output submission: {submission_path}')

In [ ]:
def load_and_flatten_jsonl(path: Path) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    records = []
    for _, row in df.iterrows():
        session = row['session']
        for event in row['events']:
            if event.get('type') == 'clicks':
                records.append({'session_id': session, 'ts': event.get('ts'), 'aid': event.get('aid')})
    return pd.DataFrame(records)

def prepare_test_sessions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['session_id', 'ts']).reset_index(drop=True)
    df['aid'] = df['aid'].astype(int) + 1
    return df


# Load test data and prepare session histories
print('Loading test data...')
test_df = load_and_flatten_jsonl(test_jsonl_path)

test_df = prepare_test_sessions(test_df)
print(f"Session count: {test_df['session_id'].nunique() if len(test_df) > 0 else 0}")
print(f'Test rows: {len(test_df)}')

In [ ]:
def build_test_session_list(df: pd.DataFrame) -> list[tuple[int, list[int]]]:
    sessions = []
    for session_id, group in df.groupby('session_id'):
        aids = group['aid'].astype(int).tolist()
        sessions.append((session_id, aids))
    return sessions



test_sessions = build_test_session_list(test_df)
print('Example session:', test_sessions[0] if test_sessions else None)
print(f'Loaded {len(test_sessions)} test sessions')

In [ ]:
from functools import partial
from pathlib import Path
import numpy as np
import pytorch_lightning as pl_lightning
import torch.nn as nn
from torch.nn import Dropout

def multiply_head_with_embedding(prediction_head: torch.Tensor, embeddings: torch.Tensor) -> torch.Tensor:
    return prediction_head.matmul(embeddings.transpose(-1, -2))

class DynamicPositionEmbedding(nn.Module):
    def __init__(self, max_len: int, dimension: int):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.register_buffer('pos_indices_const', torch.arange(0, max_len, dtype=torch.int))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x

class SASRec(pl_lightning.LightningModule):
    def __init__(self, hidden_size, dropout_rate, max_len, num_items, batch_size, sampling_style, topk_sampling=False, topk_sampling_k=1000, learning_rate=0.001, num_layers=2, loss='bce', bpr_penalty=None, optimizer='adam', output_bias=False, share_embeddings=True, final_activation=False):
        super().__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.future_mask = torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1)
        self.register_buffer('future_mask_const', self.future_mask)
        self.register_buffer('seq_diag_const', ~torch.diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones', torch.ones([batch_size, max_len, 1]))
        if output_bias and share_embeddings:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)
        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)
        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif (not share_embeddings) and output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=1, dim_feedforward=hidden_size, dropout=dropout_rate, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)
        self.merge_attn_mask = True
        self.final_activation = nn.Identity()
        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer_name = optimizer
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask: torch.Tensor) -> torch.Tensor:
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]
        padding_mask_broadcast = ~padding_mask.bool().unsqueeze(1)
        future_masks = self.future_mask_const[:seq_len, :seq_len].eq(float("-inf"))
        future_masks = future_masks.repeat(batch_size, 1, 1)
        merged_masks = padding_mask_broadcast | future_masks
        diag_masks = self.seq_diag_const[:seq_len, :seq_len].repeat(batch_size, 1, 1)
        return diag_masks & merged_masks

    def forward(self, item_indices: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        att_mask = self.merge_attn_masks(mask)
        items = self.item_embedding(item_indices)
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return x

    def configure_optimizers(self):
        if self.optimizer_name == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer_name == 'adagrad':
            return torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        else:
            raise ValueError('Optimizer not supported')

In [ ]:
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

model = SASRec.load_from_checkpoint(checkpoint_path, map_location='cpu')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print('Loaded model to', device)

In [ ]:
def make_input_tensors(session_aids: list[int], max_len: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    input_ids = session_aids[-max_len:]
    pad_len = max_len - len(input_ids)
    padded = [0] * pad_len + input_ids
    mask = [0.0] * pad_len + [1.0] * len(input_ids)
    return torch.tensor([padded], dtype=torch.long, device=device), torch.tensor([mask], dtype=torch.float, device=device)

def predict_topk(session_aids: list[int], k: int = 20) -> list[int]:
    item_ids, mask = make_input_tensors(session_aids, model.max_len, device)
    with torch.no_grad():
        outputs = model(item_ids, mask)
        last = outputs[:, -1, :]
        logits = multiply_head_with_embedding(last, model.output_embedding.weight)
        logits[:, 0] = -float('inf')
        _, indices = torch.topk(logits, k=k, dim=-1)
    return indices.squeeze(0).tolist()

results = []
for session_id, aids in test_sessions:
    top_items = predict_topk(aids, k=20)
    results.append({'session_id': session_id, 'predictions': ' '.join(str(i) for i in top_items)})

submission_df = pd.DataFrame(results)
submission_df.to_csv(submission_path, index=False)
print(f'Saved submission with {len(submission_df)} rows to {submission_path}')